# 🧠 CareerCompass AI Engine: Global Skill NER training (Autonomous)

This notebook trains a custom NER model to extract skills and roles. 

**Strategy:** We utilize a **Synthetic Data Augmentation** engine to bootstrap our training. This avoids dependencies on private/gated external datasets and ensures 100% reliability for the graduation project.

## Step 1: Install Dependencies

In [ ]:
!pip install -q transformers datasets evaluate seqeval accelerate

## Step 2: Synthetic Data Engine
We generate high-quality annotated sentences using real-world skill/role dictionaries.

In [ ]:
import random
import json

# توسيع القواميس لتشمل مجالات أكثر دقة (تم التركيز على الـ Backend والـ Architecture)
SKILLS = ["PHP", "Laravel", "Python", "FastAPI", "Node.js", "React", "Docker", "Kubernetes", "MySQL", "PostgreSQL", "Redis", "Elasticsearch", "RESTful APIs", "SOLID Principles", "Git", "AWS", "Machine Learning", "Vue.js", "Service Pattern", "Microservices"]
ROLES = ["Backend Developer", "Full Stack Developer", "Software Engineer", "Frontend Developer", "Data Scientist", "DevOps Engineer", "System Architect"]
EDUCATION = ["Computer Science", "BSc Computer Science", "Information Technology", "Software Engineering"]
CERTIFICATIONS = ["AWS Certified Solutions Architect", "Cisco CCNA", "PMP", "Google Cloud Professional"]

# قوالب مقسمة ديناميكياً لتسهيل وضع الـ BIO Tags
TEMPLATES = [
    [("Experienced in ", "O"), ("{skill1}", "SKILL"), (" and ", "O"), ("{skill2}", "SKILL"), (" as a ", "O"), ("{role}", "ROLE"), (".", "O")],
    [("Graduated with a degree in ", "O"), ("{edu}", "EDU"), (" specializing in ", "O"), ("{skill1}", "SKILL"), (".", "O")],
    [("Currently working as a ", "O"), ("{role}", "ROLE"), (" using ", "O"), ("{skill1}", "SKILL"), (" and ", "O"), ("{skill2}", "SKILL"), (".", "O")],
    [("Strong background in ", "O"), ("{skill1}", "SKILL"), (", ", "O"), ("{skill2}", "SKILL"), (", and project management.", "O")],
    [("Holding an ", "O"), ("{cert}", "CERT"), (" certification.", "O")],
    [("Passionate ", "O"), ("{role}", "ROLE"), (" with expertise in ", "O"), ("{skill1}", "SKILL"), (".", "O")]
]

# خريطة الـ Labels الجديدة
label2id = {
    "O": 0, 
    "B-SKILL": 1, "I-SKILL": 2, 
    "B-ROLE": 3, "I-ROLE": 4, 
    "B-EDU": 5, "I-EDU": 6, 
    "B-CERT": 7, "I-CERT": 8
}

def generate_bio_sample():
    template = random.choice(TEMPLATES)
    
    entities_map = {
        "{skill1}": random.choice(SKILLS),
        "{skill2}": random.choice(SKILLS),
        "{role}": random.choice(ROLES),
        "{edu}": random.choice(EDUCATION),
        "{cert}": random.choice(CERTIFICATIONS)
    }
    
    tokens = []
    ner_tags = []
    
    for text_part, tag_type in template:
        actual_text = entities_map.get(text_part, text_part)
        words = actual_text.strip().split()
        
        if not words:
            continue
            
        for i, word in enumerate(words):
            # تنظيف الكلمة من الفواصل لتجنب التشويش
            clean_word = word.replace(".", "").replace(",", "")
            if not clean_word: continue
                
            tokens.append(clean_word)
            if tag_type == "O":
                ner_tags.append(label2id["O"])
            else:
                # إذا كانت أول كلمة في الكيان نضع B-، وإلا نضع I-
                if i == 0:
                    ner_tags.append(label2id[f"B-{tag_type}"])
                else:
                    ner_tags.append(label2id[f"I-{tag_type}"])
                    
    return {"tokens": tokens, "ner_tags": ner_tags}

# رفع حجم البيانات إلى 50,000 للتدريب و 5,000 للتقييم لضمان دقة أعلى
train_data = [generate_bio_sample() for _ in range(50000)]
val_data = [generate_bio_sample() for _ in range(5000)]

with open('train.json', 'w') as f:
    for entry in train_data: f.write(json.dumps(entry) + '\n')
with open('val.json', 'w') as f:
    for entry in val_data: f.write(json.dumps(entry) + '\n')
    
print(f"✅ Generated {len(train_data)} training samples and {len(val_data)} validation samples with BIO Tags.")

## Step 3: Load Data for Transformers

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
import numpy as np
import evaluate

dataset = load_dataset('json', data_files={'train': 'train.json', 'test': 'val.json'})

# تحديث قائمة الـ Labels لتعكس نظام الـ BIO الجديد
label_list = ["O", "B-SKILL", "I-SKILL", "B-ROLE", "I-ROLE", "B-EDU", "I-EDU", "B-CERT", "I-CERT"]
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

# سنبقي على النموذج الحالي في هذه الخلية مؤقتاً لتجهيز البيانات
MODEL_CHECKPOINT = "distilbert-base-cased"
# MODEL_CHECKPOINT = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None: 
                label_ids.append(-100) # تجاهل المسافات والرموز الخاصة
            elif word_idx != previous_word_idx: 
                label_ids.append(label[word_idx]) # تعيين التاج للكلمة
            else: 
                label_ids.append(-100) # تجاهل الأجزاء الفرعية للكلمة (Subwords)
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_ds = dataset.map(tokenize_and_align_labels, batched=True)
print("✅ Transformers tokenizer and labels configured successfully.")

## Step 4: Training & Metrics

In [ ]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
import numpy as np
import evaluate

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT, 
    num_labels=len(label_list), 
    id2label=id2label, 
    label2id=label2id
)
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    
    # Remove ignored index (-100) and map to string labels
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    
    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"]
    }

args = TrainingArguments(
    "career-compass-ner",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    push_to_hub=False,
    fp16=True  # Enables mixed precision training for faster execution on T4 GPU
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    processing_class=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics
)

trainer.train()

## Step 5: Save Model Weights

In [ ]:
trainer.save_model("career_compass_ner_final")
tokenizer.save_pretrained("career_compass_ner_final")
print("✅ Training Complete! Model and Tokenizer saved successfully.")